# Public-transport weekday stop frequencies

Combines the representative Monday–Friday school and holiday extracts with the supplied stop-location workbook. The average is unweighted across the two day types. The frequency exports omit stops with zero departures, so omitted rows are explicitly set to zero. `weekday_line_count` is the number of distinct passenger-facing lines appearing in either weekday extract; grouping by station and line removes direction-level duplicates. `weekday_route_ids` stores the direction-deduplicated technical route keys with timetable-version suffixes removed; missing 2016 keys use a `TEXT:` fallback.

**Coordinate note:** `MRCV-x` and `MRCV-y2` are Web-Mercator coordinates. `MRCV-y` is an internal positive representation and is not used.

In [ ]:
import os
import re
from pathlib import Path

os.chdir(r"D:\CO2_Masterarbeit\CO2_Masterarbeit")

import geopandas as gpd
import pandas as pd

DATA_DIR = Path("OGD/Public_Transport/Verbund_Daten_work")
OUTPUT_DIR = Path("OGD/Public_Transport")
YEARS = (2016, 2017, 2018, 2019, 2020, 2022)
EFA_PREFIX = 63_200_000

In [ ]:
def read_frequency_csv(path):
    for encoding in ("utf-8", "cp1252", "latin1"):
        try:
            return pd.read_csv(path, encoding=encoding, dtype={"EFA-Haltestellennummer": "string"})
        except UnicodeDecodeError:
            continue
    raise ValueError(f"Could not decode {path}")


def source_file(year, day_type, table_number=1):
    files = list(DATA_DIR.glob(f"Manfred_Brandl_stv-{year}*/**/Montag-Freitag-{day_type}_*.Datei{table_number}_ab.csv"))
    if len(files) != 1:
        raise FileNotFoundError(f"Expected one {day_type} weekday Datei{table_number} table for {year}; found {files}")
    return files[0]


def frequency_table(year, day_type, column):
    frequency = read_frequency_csv(source_file(year, day_type)).rename(columns={
        "EFA-Haltestellennummer": "efa_stop_id",
        "GlobalID": "global_id",
        "Haltestellenname mit Ort": "station_name",
        "Anzahl Abfahrten": column,
    })
    frequency["efa_stop_id"] = pd.to_numeric(frequency["efa_stop_id"], errors="raise").astype("int64")
    frequency["station_id"] = frequency["efa_stop_id"] - EFA_PREFIX
    frequency[column] = pd.to_numeric(frequency[column], errors="raise")
    frequency = frequency[["station_id", "efa_stop_id", "global_id", "station_name", column]]
    if frequency["station_id"].duplicated().any():
        raise ValueError(f"Duplicate station IDs in {year} {day_type}")
    return frequency


def weekday_line_table(year):
    line_tables = []
    for day_type in ("Schule", "Ferien"):
        lines = read_frequency_csv(source_file(year, day_type, table_number=2))
        line_key_column = next(column for column in lines.columns if column.startswith("Linienschl"))
        lines = lines.rename(columns={
            "EFA-Haltestellennummer": "efa_stop_id",
            "Linientext": "line_name",
            line_key_column: "line_key",
        })[["efa_stop_id", "line_key", "line_name"]]
        lines["efa_stop_id"] = pd.to_numeric(lines["efa_stop_id"], errors="raise").astype("int64")
        lines["station_id"] = lines["efa_stop_id"] - EFA_PREFIX
        lines["line_name"] = lines["line_name"].astype("string").str.strip()
        lines["route_id"] = lines["line_key"].astype("string").str.strip().str.replace(r"\..*$", "", regex=True)
        missing_route_id = lines["route_id"].isna() | lines["route_id"].eq("")
        fallback = "TEXT:" + lines["line_name"].str.upper().str.replace(r"\s+", "", regex=True)
        lines.loc[missing_route_id, "route_id"] = fallback.loc[missing_route_id]
        line_tables.append(lines.dropna(subset=["line_name"]))

    # Datei2 is aggregated by station and line, not direction. The explicit
    # station-line deduplication also removes duplicates across day types.
    station_lines = pd.concat(line_tables, ignore_index=True)
    station_lines = station_lines[station_lines["line_name"].ne("")]
    passenger_lines = station_lines.drop_duplicates(["station_id", "line_name"])
    line_counts = (passenger_lines.groupby("station_id", as_index=False).size()
                   .rename(columns={"size": "weekday_line_count"}))
    route_ids = (station_lines.drop_duplicates(["station_id", "route_id"])
                 .groupby("station_id")["route_id"]
                 .agg(lambda values: "|".join(sorted(values)))
                 .rename("weekday_route_ids").reset_index())
    return line_counts.merge(route_ids, on="station_id", how="outer", validate="one_to_one")

In [ ]:
locations = pd.read_excel(DATA_DIR / "Manfred_Brandl_Haltestellen-Mangurano.xlsx", sheet_name="Haltestellen")
locations = locations.rename(columns={
    "Nummer": "station_id",
    "Name mit Ort": "station_name_location_file",
    "MRCV-x": "x_web_mercator",
    "MRCV-y2": "y_web_mercator",
})
locations = locations[["station_id", "station_name_location_file", "x_web_mercator", "y_web_mercator"]].copy()
locations["station_id"] = pd.to_numeric(locations["station_id"], errors="raise").astype("int64")
locations[["x_web_mercator", "y_web_mercator"]] = locations[["x_web_mercator", "y_web_mercator"]].apply(pd.to_numeric, errors="coerce")
locations = locations.dropna(subset=["x_web_mercator", "y_web_mercator"])
assert not locations["station_id"].duplicated().any()
print(f"Located stops in workbook: {len(locations):,}")

In [ ]:
datasets = {}
validation_rows = []

for year in YEARS:
    school = frequency_table(year, "Schule", "weekday_school_departures")
    holiday = frequency_table(year, "Ferien", "weekday_holiday_departures")
    frequencies = school.merge(holiday[["station_id", "weekday_holiday_departures"]], on="station_id", how="outer", validate="one_to_one")
    frequencies = frequencies.merge(weekday_line_table(year), on="station_id", how="left", validate="one_to_one")
    frequencies["school_source_has_stop"] = frequencies["weekday_school_departures"].notna()
    frequencies["holiday_source_has_stop"] = frequencies["weekday_holiday_departures"].notna()
    frequencies[["weekday_school_departures", "weekday_holiday_departures"]] = frequencies[["weekday_school_departures", "weekday_holiday_departures"]].fillna(0)
    frequencies["weekday_line_count"] = frequencies["weekday_line_count"].fillna(0).astype("int64")
    frequencies["weekday_avg_departures"] = frequencies[["weekday_school_departures", "weekday_holiday_departures"]].mean(axis=1)

    dataset = frequencies.merge(locations, on="station_id", how="left", validate="one_to_one")
    dataset["year"] = year
    dataset["location_matched"] = dataset["x_web_mercator"].notna() & dataset["y_web_mercator"].notna()
    dataset = gpd.GeoDataFrame(dataset, geometry=gpd.points_from_xy(dataset["x_web_mercator"], dataset["y_web_mercator"]), crs="EPSG:3857")
    dataset = dataset[["year", "station_id", "efa_stop_id", "global_id", "station_name", "station_name_location_file",
                       "weekday_school_departures", "weekday_holiday_departures", "weekday_avg_departures", "weekday_line_count", "weekday_route_ids",
                       "school_source_has_stop", "holiday_source_has_stop", "location_matched",
                       "x_web_mercator", "y_web_mercator", "geometry"]]
    dataset.to_parquet(OUTPUT_DIR / f"public_transport_weekday_stop_frequency_{year}.geoparquet", index=False)
    datasets[year] = dataset
    validation_rows.append({
        "year": year,
        "frequency_stops": len(dataset),
        "coordinate_matches": dataset["location_matched"].sum(),
        "unmatched_coordinates": (~dataset["location_matched"]).sum(),
        "school_source_rows_omitted_zero_departures": (~dataset["school_source_has_stop"]).sum(),
        "holiday_source_rows_omitted_zero_departures": (~dataset["holiday_source_has_stop"]).sum(),
    })

In [ ]:
validation = pd.DataFrame(validation_rows)
validation.to_csv(OUTPUT_DIR / "public_transport_weekday_frequency_validation.csv", index=False)

summary = pd.DataFrame([{
    "year": year,
    "stops": len(dataset),
    "located_stops": dataset["location_matched"].sum(),
    "total_avg_departures": dataset["weekday_avg_departures"].sum(),
    "median_avg_departures": dataset["weekday_avg_departures"].median(),
    "median_weekday_line_count": dataset["weekday_line_count"].median(),
    "max_weekday_line_count": dataset["weekday_line_count"].max(),
} for year, dataset in datasets.items()]).sort_values("year")
summary.to_csv(OUTPUT_DIR / "public_transport_weekday_frequency_summary.csv", index=False)

assert all(dataset.crs.to_epsg() == 3857 for dataset in datasets.values())
assert all((dataset["weekday_avg_departures"] == (dataset["weekday_school_departures"] + dataset["weekday_holiday_departures"]) / 2).all() for dataset in datasets.values())
validation
summary

In [ ]:
dataset[dataset.station_name=="Bahnhof Leoben"]

## 2025 stop locations and weekday frequencies

The 2025 stop shapefile already includes weekday school and holiday departure counts and a comma-separated `LINIE` field. This export standardizes it to the same GeoParquet schema and Web-Mercator CRS as the earlier yearly files. Exact duplicate line identifiers are counted once per station.

In [ ]:
STOPS_2025_ZIP = Path("OGD/Public_Transport/Haltestellen_2025.zip")
STOPS_2025_OUTPUT = STOPS_2025_ZIP.with_name("public_transport_weekday_stop_frequency_2025.geoparquet")

stops_2025 = gpd.read_file(f"zip://{STOPS_2025_ZIP.resolve()}")
if stops_2025.crs is None:
    raise ValueError("The 2025 stops shapefile has no coordinate reference system.")

stops_2025 = stops_2025.to_crs("EPSG:3857")
stops_2025["station_id"] = pd.to_numeric(stops_2025["HNR"], errors="raise").astype("int64")
stops_2025["efa_stop_id"] = stops_2025["station_id"] + EFA_PREFIX
stops_2025["global_id"] = "at:46:" + stops_2025["station_id"].astype("string")
stops_2025["station_name"] = stops_2025["HNAME_LANG"].astype("string")
stops_2025["station_name_location_file"] = stops_2025["station_name"]
stops_2025["weekday_school_departures"] = pd.to_numeric(stops_2025["MoFr_S"], errors="raise")
stops_2025["weekday_holiday_departures"] = pd.to_numeric(stops_2025["MoFr_F"], errors="raise")
stops_2025["weekday_avg_departures"] = stops_2025[["weekday_school_departures", "weekday_holiday_departures"]].mean(axis=1)
stops_2025["weekday_line_count"] = stops_2025["LINIE"].fillna("").map(
    lambda value: len({line.strip() for line in str(value).split(",") if line.strip()})
).astype("int64")
stops_2025["weekday_route_ids"] = stops_2025["LINIE"].fillna("").map(
    lambda value: "|".join(sorted({line.strip() for line in str(value).split(",") if line.strip()}))
).astype("string")
stops_2025["school_source_has_stop"] = True
stops_2025["holiday_source_has_stop"] = True
stops_2025["location_matched"] = stops_2025.geometry.notna()
stops_2025["x_web_mercator"] = stops_2025.geometry.x
stops_2025["y_web_mercator"] = stops_2025.geometry.y
stops_2025["year"] = 2025

stops_2025 = stops_2025[[
    "year", "station_id", "efa_stop_id", "global_id", "station_name", "station_name_location_file",
    "weekday_school_departures", "weekday_holiday_departures", "weekday_avg_departures", "weekday_line_count", "weekday_route_ids",
    "school_source_has_stop", "holiday_source_has_stop", "location_matched",
    "x_web_mercator", "y_web_mercator", "geometry",
]]
assert stops_2025.crs.to_epsg() == 3857
assert (stops_2025["weekday_avg_departures"] == (stops_2025["weekday_school_departures"] + stops_2025["weekday_holiday_departures"]) / 2).all()
stops_2025.to_parquet(STOPS_2025_OUTPUT, index=False)
print(f"Wrote {len(stops_2025):,} stops to {STOPS_2025_OUTPUT}")
stops_2025.head()

## Interactive station map for one year

Set `MAP_YEAR` to one of the generated source years. The Leaflet map contains every station with valid coordinates and shows its weekday departures and direction-deduplicated line count. The resulting HTML file is standalone apart from the web map tiles.

In [ ]:
import json

import folium

MAP_YEAR = 2025
MAP_OUTPUT_DIR = Path("FIG/Haltestellen")
MAP_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

map_path = OUTPUT_DIR / f"public_transport_weekday_stop_frequency_{MAP_YEAR}.geoparquet"
map_stops = gpd.read_parquet(map_path).dropna(subset=["geometry"]).to_crs("EPSG:4326")
map_stops = map_stops[["station_id", "station_name", "weekday_avg_departures", "weekday_line_count", "geometry"]].copy()
map_center = [map_stops.geometry.y.median(), map_stops.geometry.x.median()]

station_map = folium.Map(location=map_center, zoom_start=8, tiles="CartoDB positron", control_scale=True)
folium.GeoJson(
    data=json.loads(map_stops.to_json()),
    name=f"Public-transport stations {MAP_YEAR}",
    marker=folium.CircleMarker(radius=3, color="#1f4e79", weight=1, fill=True, fill_opacity=0.75),
    tooltip=folium.GeoJsonTooltip(
        fields=["station_name", "weekday_line_count", "weekday_avg_departures"],
        aliases=["Station:", "Weekday lines:", "Average weekday departures:"],
        localize=True,
    ),
    popup=folium.GeoJsonPopup(
        fields=["station_id", "station_name", "weekday_line_count", "weekday_avg_departures"],
        aliases=["Station ID:", "Station:", "Weekday lines:", "Average weekday departures:"],
        localize=True,
    ),
).add_to(station_map)
folium.LayerControl().add_to(station_map)

map_output = MAP_OUTPUT_DIR / f"public_transport_stations_{MAP_YEAR}_interactive.html"
station_map.save(map_output)
print(f"Mapped {len(map_stops):,} stations and wrote {map_output}")
station_map

## Rail-station candidates for routing (2015–2025)

These layers are extracts of the public-transport stop files above, not OSM data. The station-name rule below retains likely railway stops, clips them to Styria plus a 5 km buffer, and writes them to `Rail stations/`. Years without an observed stop file use the nearest available source; 2021 combines matching 2020 and 2022 stops.

In [ ]:
RAIL_OUTPUT_DIR = Path(r'OGD/Public_Transport/Rail stations')
MUNICIPALITIES_PATH = Path(r'OGD/Gemeindegrenzen.zip')
PT_SOURCE_YEARS = {2015: (2016,), 2016: (2016,), 2017: (2017,), 2018: (2018,), 2019: (2019,), 2020: (2020,), 2021: (2020, 2022), 2022: (2022,), 2023: (2022,), 2024: (2025,), 2025: (2025,)}
TRAIN_STATION_NAME_REGEX = r'(?ix)^(?!.*\b(?:BH|WP|AO)\b)(?!.*(?:Busbahnhof|Fernbusbahnhof|Güterbahnhof|Verschiebebahnhof))(?!.*\bAbzw\.?\s+.*Bahnhof\b)(?!.*\bBahnhof(?:straße|str\.?|weg|gürtel)\b)(?!.*\b(?:ehemaliger|ehem\.?|Alter)\s+Bahnhof\b)(?!.*\b(?:Seilbahn|Gondelbahn|Gondlbahn|Sesselbahn|Schloßbergbahn|Autobahn)\b)(?!.*\b(?:Bahndurchlass|Bahnwärterhaus|Bahndamm|Bahnweg)\b)(?!.*(?:\[ALT\]|\(ALT\)|ersetzt\s+durch|\bPLAN\b))(?=.*(?:Bahnhof\b|Bahnhaltestelle\b|\bHbf\b|\bBf\b|\bS[- ]?Bahn\b)).+$'

def assigned_stops(year):
    frames = []
    for source_year in PT_SOURCE_YEARS[year]:
        path = Path(r'OGD/Public_Transport') / f'public_transport_weekday_stop_frequency_{source_year}.geoparquet'
        frame = gpd.read_parquet(path).to_crs('EPSG:4326').dropna(subset=['station_id', 'geometry']).copy()
        frame['pt_source_year'] = source_year
        frame['source_stop_id'] = frame['station_id'].astype(str)
        frames.append(frame)
    if len(frames) == 1:
        stops = frames[0]
        stops['source_years'] = str(PT_SOURCE_YEARS[year][0])
        stops['source_stop_ids'] = stops['source_stop_id']
        stops['imputation_method'] = 'observed' if year == PT_SOURCE_YEARS[year][0] else f'nearest_observed_year_{PT_SOURCE_YEARS[year][0]}'
    else:
        combined = pd.concat(frames, ignore_index=True)
        rows = []
        for station_id, group in combined.groupby('station_id', sort=True):
            row = group.sort_values('pt_source_year').iloc[0].copy()
            row['weekday_school_departures'] = pd.to_numeric(group['weekday_school_departures'], errors='coerce').mean()
            row['weekday_holiday_departures'] = pd.to_numeric(group['weekday_holiday_departures'], errors='coerce').mean()
            row['weekday_avg_departures'] = (row['weekday_school_departures'] + row['weekday_holiday_departures']) / 2
            row['source_years'] = '|'.join(map(str, sorted(group['pt_source_year'].unique())))
            row['source_stop_ids'] = '|'.join(f'{r.pt_source_year}:{r.station_id}' for _, r in group.sort_values('pt_source_year').iterrows())
            row['imputation_method'] = 'mean_2020_2022_matching_station_id' if len(group) == 2 else '2021_one_sided_stop_retained'
            rows.append(row)
        stops = gpd.GeoDataFrame(rows, geometry='geometry', crs='EPSG:4326')
    stops['year'] = year
    return stops

municipalities = gpd.read_file(f'zip://{MUNICIPALITIES_PATH.resolve().as_posix()}').to_crs('EPSG:3035')
study_area = gpd.GeoSeries([municipalities.union_all().buffer(5_000)], crs='EPSG:3035').to_crs('EPSG:4326').iloc[0]
RAIL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for year in range(2015, 2026):
    candidates = assigned_stops(year)
    candidates = candidates[candidates.geometry.within(study_area)].copy()
    candidates['train_station_name_regex'] = TRAIN_STATION_NAME_REGEX
    candidates['train_station_candidate'] = candidates['station_name'].fillna('').astype(str).str.contains(re.compile(TRAIN_STATION_NAME_REGEX), na=False)
    candidates = candidates[candidates['train_station_candidate']].copy()
    output = RAIL_OUTPUT_DIR / f'austria-{year}-rail_station_candidates.geoparquet'
    candidates.to_parquet(output, index=False)
    print(f'Wrote {len(candidates):,} rail-station candidates to {output.name}')
